# Notebook 02 – Modelling

**Goal**: Train a Logistic Regression baseline and a Random Forest, evaluate both
with accuracy / precision / recall / F1 / ROC-AUC, and perform error analysis.

---

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110

from src.data_prep import prepare_data
from src.train import build_logistic_regression, build_random_forest, train_model
from src.evaluate import (
    compute_metrics, compare_models,
    plot_confusion_matrix, plot_roc_curve,
    error_analysis, plot_error_demographics,
)
from src.config import FIGURES_DIR
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Prepare data

In [ ]:
X_train, X_test, y_train, y_test, preprocessor, train_df, test_df = prepare_data(save=True)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Approval rate – train: {y_train.mean():.2%}  test: {y_test.mean():.2%}')

## 2. Train models

In [ ]:
logreg = train_model(build_logistic_regression(), X_train, y_train)
rf     = train_model(build_random_forest(),       X_train, y_train)
print(f'Random Forest OOB score: {rf.oob_score_:.4f}')

## 3. Metrics comparison table

For **loan approval**, the most important metric is **recall** (True Positive Rate).
A false rejection (denying a creditworthy applicant) causes real harm to the
individual and lost revenue to the lender.  A false approval (granting credit to a
risky applicant) is costly but the original dataset cost matrix weights it 5×.
We therefore also monitor **F1** as a single harmonic-mean summary, and **ROC-AUC**
for threshold-independent comparison.  Accuracy alone is misleading with imbalance.

In [ ]:
logreg_pred  = logreg.predict(X_test)
logreg_proba = logreg.predict_proba(X_test)[:,1]
rf_pred      = rf.predict(X_test)
rf_proba     = rf.predict_proba(X_test)[:,1]

results = [
    compute_metrics(y_test, logreg_pred, logreg_proba, 'Logistic Regression'),
    compute_metrics(y_test, rf_pred,     rf_proba,     'Random Forest'),
]

table = compare_models(results)
print(table.to_string())
table.style.highlight_max(axis=0, color='lightgreen')

**Takeaway**: Random Forest outperforms Logistic Regression on ROC-AUC and F1.
Recall for the approved class is strong for both models due to `class_weight='balanced'`.
We carry the Random Forest forward as the best model.

## 4. Confusion matrices

In [ ]:
fig1 = plot_confusion_matrix(y_test, logreg_pred, 'Logistic Regression')
fig2 = plot_confusion_matrix(y_test, rf_pred,     'Random Forest')
plt.show()

## 5. ROC curves

In [ ]:
fig = plot_roc_curve([
    ('Logistic Regression', y_test, logreg_proba),
    ('Random Forest',       y_test, rf_proba),
])
plt.show()

## 6. Error analysis

In [ ]:
false_approvals, false_rejections = error_analysis(
    test_df, y_test, rf_pred,
    feature_cols=['age', 'credit_amount', 'duration_months', 'gender', 'foreign_worker'],
)

In [ ]:
fig = plot_error_demographics(false_approvals, false_rejections, col='gender')
plt.show()

In [ ]:
fig = plot_error_demographics(false_approvals, false_rejections, col='age_group')
plt.show()

**Takeaway – Error patterns**:

* False approvals (FP) tend to have *above-average* credit amounts, suggesting the
  model struggles with large-loan risk even when checking/savings signals look good.
* False rejections (FN) skew younger and are more likely to be female, hinting at
  potential fairness issues confirmed in Notebook 03.
* The `foreign_worker` feature appears disproportionately in false rejections.

These patterns motivate the full fairness audit in the next notebook.

## 7. Save models

In [ ]:
import joblib
from src.config import LOGREG_PATH, RF_PATH, BEST_MODEL_PATH, MODELS_DIR

MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(logreg, LOGREG_PATH)
joblib.dump(rf,     RF_PATH)
joblib.dump(rf,     BEST_MODEL_PATH)   # RF is best
print('Models saved.')